# LaBSE — Hebrew–English employer-name equivalence probe

**Purpose:** Test whether the *pretrained* LaBSE model assigns higher cross-lingual similarity to corresponding organization names than to unrelated organizations. No retraining, paid API, token, GPU, or manually maintained dictionary is required. Run every cell in **Google Colab (free CPU)**.

**Limitations:** LaBSE is trained for cross-language sentence representations, **not** verified employer entity resolution. This notebook reports observed cosine similarities and nearest neighbors. A high score is not proof that names refer to the same legal entity. Real data integration requires validation. The first execution downloads publicly available model weights (~1.9 GB) and requires Internet access. No model inference result is claimed until this notebook executes.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

import torch
from sentence_transformers import SentenceTransformer
import numpy as np

print("PyTorch:", torch.__version__)
print("Device: CPU (no GPU required)")

In [ ]:
names = [
    "Intel", "אינטל", "NVIDIA", "נבידיה",
    "Microsoft", "מיקרוסופט", "bird", "birds", "ציפור",
    "קרית גת", "קריית גת", "בית ספר יסודי", "primary school"
]

pairs = [
    ("Intel", "אינטל", "company alias, English–Hebrew"),
    ("NVIDIA", "נבידיה", "company alias, English–Hebrew"),
    ("Microsoft", "מיקרוסופט", "company alias, English–Hebrew"),
    ("Intel", "NVIDIA", "different companies, control"),
    ("Intel", "בית ספר יסודי", "unrelated, control"),
    ("bird", "birds", "singular/plural"),
    ("bird", "ציפור", "translation"),
    ("קרית גת", "קריית גת", "same locality, spelling variants"),
    ("בית ספר יסודי", "primary school", "phrase translation"),
]

torch.set_num_threads(2)
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
vectors = model.encode(names, batch_size=4, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
assert vectors.shape == (len(names), 768), vectors.shape
assert np.isfinite(vectors).all(), "Nonfinite values in embeddings"
positions = {name: i for i, name in enumerate(names)}

print(f"Encoded {len(names)} labels into {vectors.shape[1]}-D vectors")
print("Pairwise cosine similarities (observed, not predetermined):")
for left, right, group in pairs:
    sim = float(vectors[positions[left]] @ vectors[positions[right]])
    print(f"{left!r:>22} <=> {right!r:<22} cosine={sim:+.4f} | {group}")

print("\nNearest neighbor for each organization name (excluding itself):")
for name in names[:6]:
    sims = vectors @ vectors[positions[name]]
    sims[positions[name]] = -np.inf
    idx = int(np.argmax(sims))
    print(f"{name!r} -> {names[idx]!r} (cosine={float(sims[idx]):+.4f})")

print("\nCompleted: model downloaded, PyTorch inference executed, and scores computed.")

**Interpretation rule:** A successful run proves that the pipeline produces real embeddings and similarity scores. It does **not** establish the precision of multilingual company-name matching. Decide whether to use this representation only after evaluating positive and negative pairs from the actual `ShemAvoda` distribution, including ambiguous names and generic employer descriptions. The notebook intentionally performs no automatic name replacement.